In [36]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.tree import export_text

In [22]:
# import data
df = pd.read_csv('~/python/data/ml_zoomcamp/car_fuel_efficiency_2026.csv')
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


### Data Preparation

In [23]:
# fill missing values with zero
df.fillna(0, inplace=True)
print(f'Null Values:\n{df.isnull().sum()}')

Null Values:
model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64


In [24]:
# splitting data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# target variable
y_train = df_train['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']

# remove target from the train, val, and test sets
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [25]:
# One hot encoding
train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test.to_dict(orient='records')
X_test = dv.transform(test_dicts)

### Question 1


Train a model with max_depth=1. What original feature is used for spitting the data?

In [28]:
# train decision stump
tree = DecisionTreeRegressor(max_depth=1)
tree.fit(X_train, y_train)

# view the tree
print(export_text(tree, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



'model_year' is the original feature that splits the data.

### Question 2


Train a random forest regressor with
- n_estimators = 10
- random_state = 1
- n_jobs = -1


What is the RMSE of this model on the validation data?

In [32]:
# train forest regressor
forest = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)
forest.fit(X_train, y_train)

# validate model
val_pred = forest.predict(X_val)
score = root_mean_squared_error(y_val, val_pred)
print(f'RMSE of Random Forest Regressor: {score}')

RMSE of Random Forest Regressor: 1.8332197358745623


### Question 3


Optimise the **n_estimators** hyperparameter with
- n_estimators = [10, 50, 100, 150]


Which value gives the lowest RMSE?

In [38]:
# loop over estimators
estimators = [10, 50, 100, 150]

for e in estimators:
    forest = RandomForestRegressor(n_estimators=e, random_state=1)
    forest.fit(X_train, y_train)

    val_preds = forest.predict(X_val)
    score = root_mean_squared_error(y_val, val_preds)

    # print results
    print(f'RMSE of Forest with {e} Estimators: {np.round(score, 3)}')

RMSE of Forest with 10 Estimators: 1.833
RMSE of Forest with 50 Estimators: 1.774
RMSE of Forest with 100 Estimators: 1.77
RMSE of Forest with 150 Estimators: 1.769


### Question 4

In [39]:
# outer loop is max_depth
depths = [10, 15, 20, 25]
estimators = [10, 50, 100, 150]

for d in depths:
    scores = []
    for e in estimators:
        forest = RandomForestRegressor(n_estimators=e, max_depth=d, random_state=1)
        forest.fit(X_train, y_train)

        val_preds = forest.predict(X_val)
        score = root_mean_squared_error(y_val, val_preds)
        scores.append(score)

    # print results
    print(f'RMSE of Forest with Depth {d}: {np.round(np.mean(scores), 3)}')

RMSE of Forest with Depth 10: 1.754
RMSE of Forest with Depth 15: 1.784
RMSE of Forest with Depth 20: 1.787
RMSE of Forest with Depth 25: 1.786


### Question 5

In [54]:
# train the model
forest = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
forest.fit(X_train, y_train)

# get most important feature
feature_importance = forest.feature_importances_
features = dv.get_feature_names_out()
feature_importance_df = pd.DataFrame(
    {
        'Feature': features,
        'Importance': feature_importance
    }
)
feature_importance_df

,Feature,Importance
0,acceleration,0.054998
1,drivetrain=All-wheel drive,0.007951
2,drivetrain=Front-wheel drive,0.020062
3,drivetrain=Rear-wheel drive,0.005638
4,engine_displacement,0.061281
5,fuel_type=Diesel,0.016571
6,fuel_type=Gasoline,0.160712
7,fuel_type=Hybrid,0.016272
8,horsepower,0.078271
9,model_year,0.296107


### Question 6

In [55]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [56]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100)

y_pred = model.predict(dval)
root_mean_squared_error(y_val, y_pred)

1.8264579799904583

In [57]:
xgb_params = {
    'eta': 0.1,
    'max_depth': 6,
    'min_child_weight': 1,

    'objective': 'reg:squarederror',
    'nthread': 8,

    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100)

y_pred = model.predict(dval)
root_mean_squared_error(y_val, y_pred)

1.724810059827976